# Notebook 13: Adiabatic Quantum Computing

**Series: Quantum Computing from Ground Up**

---

## Learning Objectives

1. State and understand the adiabatic theorem
2. Master the adiabatic quantum computation (AQC) model
3. Encode optimization problems as Hamiltonians
4. Understand the energy gap and its role in computation time
5. Prove equivalence between AQC and the gate model
6. Connect to quantum annealing (D-Wave)
7. Simulate adiabatic evolution for optimization problems

## 1. The Adiabatic Theorem

### 1.1 Statement

The **adiabatic theorem** (Born and Fock, 1928) is one of the most important results in quantum mechanics:

> A quantum system that starts in the ground state of a slowly-varying Hamiltonian will remain in the instantaneous ground state, provided the evolution is sufficiently slow.

Formally, consider a time-dependent Hamiltonian $H(s)$ with $s = t/T \in [0, 1]$ where $T$ is the total evolution time. Let $|E_0(s)\rangle$ be the instantaneous ground state with energy $E_0(s)$, and let the **energy gap** be:

$$\Delta(s) = E_1(s) - E_0(s)$$

where $E_1(s)$ is the first excited energy.

### 1.2 Adiabatic Condition

The system remains in the ground state with high probability if:

$$T \gg \frac{\max_s \left|\langle E_1(s) | \frac{dH}{ds} | E_0(s) \rangle\right|}{\min_s \Delta(s)^2}$$

A simpler sufficient condition:

$$\boxed{T \gg \frac{\|dH/ds\|}{\Delta_{\min}^2}}$$

where $\Delta_{\min} = \min_{s \in [0,1]} \Delta(s)$ is the **minimum energy gap**.

### 1.3 Proof Sketch

Working in the instantaneous eigenbasis $\{|E_n(s)\rangle\}$, the state $|\psi(s)\rangle = \sum_n c_n(s) e^{i\phi_n(s)} |E_n(s)\rangle$ satisfies:

$$\dot{c}_m = -c_m \langle E_m | \dot{E}_m \rangle - \sum_{n \neq m} c_n \frac{\langle E_m | \dot{H} | E_n \rangle}{E_n - E_m} e^{i(\phi_n - \phi_m)}$$

where the dynamic phase is $\phi_n(s) = -T \int_0^s E_n(s')\,ds'$. The adiabatic approximation neglects the off-diagonal terms, valid when $T$ is large and $\Delta$ is bounded away from zero.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm, eigh

# Demonstrate the adiabatic theorem with a simple 2-level system
# H(s) = (1-s)*H_initial + s*H_final
# H_initial = -sigma_x, H_final = -sigma_z

sigma_x = np.array([[0, 1], [1, 0]], dtype=complex)
sigma_z = np.array([[1, 0], [0, -1]], dtype=complex)
I2 = np.eye(2, dtype=complex)

def H_adiabatic(s, H_i, H_f):
    """Time-dependent Hamiltonian H(s) = (1-s)*H_i + s*H_f"""
    return (1 - s) * H_i + s * H_f

def compute_spectrum(H_i, H_f, n_points=200):
    """Compute the energy spectrum along the adiabatic path."""
    s_vals = np.linspace(0, 1, n_points)
    energies = []
    gaps = []
    
    for s in s_vals:
        H = H_adiabatic(s, H_i, H_f)
        eigvals = np.sort(np.real(np.linalg.eigvalsh(H)))
        energies.append(eigvals)
        if len(eigvals) >= 2:
            gaps.append(eigvals[1] - eigvals[0])
    
    return s_vals, np.array(energies), np.array(gaps)

# Compute spectrum
H_i = -sigma_x
H_f = -sigma_z
s_vals, energies, gaps = compute_spectrum(H_i, H_f)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Energy spectrum
for i in range(energies.shape[1]):
    ax1.plot(s_vals, energies[:, i], linewidth=2, label=f'E_{i}')
ax1.set_xlabel('s = t/T', fontsize=12)
ax1.set_ylabel('Energy', fontsize=12)
ax1.set_title('Energy Spectrum during Adiabatic Evolution', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Energy gap
ax2.plot(s_vals, gaps, 'r-', linewidth=2)
ax2.axhline(y=min(gaps), color='k', linestyle='--', alpha=0.5, label=f'Δ_min = {min(gaps):.3f}')
ax2.set_xlabel('s = t/T', fontsize=12)
ax2.set_ylabel('Energy Gap Δ(s)', fontsize=12)
ax2.set_title('Energy Gap', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Minimum gap: Δ_min = {min(gaps):.4f} at s = {s_vals[np.argmin(gaps)]:.4f}")

In [ ]:
def simulate_adiabatic_evolution(H_i, H_f, T, n_steps=1000):
    """
    Simulate adiabatic evolution using Trotterization.
    
    Parameters:
        H_i: initial Hamiltonian
        H_f: final Hamiltonian
        T: total evolution time
        n_steps: number of Trotter steps
    
    Returns:
        ground_state_prob: probability of being in ground state vs time
    """
    dim = H_i.shape[0]
    dt = T / n_steps
    
    # Initial state: ground state of H_i
    eigvals, eigvecs = eigh(H_i)
    psi = eigvecs[:, 0].astype(complex)  # Ground state
    
    ground_state_probs = []
    s_values = []
    
    for step in range(n_steps + 1):
        s = step / n_steps
        s_values.append(s)
        
        # Current ground state
        H_current = H_adiabatic(s, H_i, H_f)
        eigvals_current, eigvecs_current = eigh(H_current)
        ground = eigvecs_current[:, 0]
        
        # Ground state probability
        prob = np.abs(np.vdot(ground, psi))**2
        ground_state_probs.append(prob)
        
        # Evolve (except last step)
        if step < n_steps:
            U = expm(-1j * H_current * dt)
            psi = U @ psi
    
    return np.array(s_values), np.array(ground_state_probs)

# Compare different evolution times
fig, ax = plt.subplots(figsize=(12, 6))

for T in [0.5, 1, 2, 5, 10, 20]:
    s_vals_sim, gs_probs = simulate_adiabatic_evolution(H_i, H_f, T, n_steps=500)
    ax.plot(s_vals_sim, gs_probs, linewidth=2, label=f'T = {T}')

ax.set_xlabel('s = t/T', fontsize=12)
ax.set_ylabel('Ground State Probability', fontsize=12)
ax.set_title('Adiabatic Evolution: Effect of Total Time T', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print("Larger T → slower evolution → higher ground state fidelity")
print("The system must be slow enough to avoid Landau-Zener transitions at the minimum gap.")

## 2. Adiabatic Quantum Computation (AQC)

### 2.1 The AQC Framework

**Adiabatic Quantum Computation** solves optimization problems by:

1. **Encode** the solution as the ground state of a **problem Hamiltonian** $H_P$
2. **Start** in the easily-prepared ground state of a simple **initial Hamiltonian** $H_0$
3. **Slowly evolve** from $H_0$ to $H_P$:

$$H(s) = (1 - s)H_0 + s H_P, \quad s = t/T$$

4. By the adiabatic theorem, the system ends in the ground state of $H_P$ — which is the **solution**!

### 2.2 Standard Initial Hamiltonian

The standard choice is the **transverse field**:

$$H_0 = -\sum_{i=1}^n X_i$$

Its ground state is $|+\rangle^{\otimes n} = \frac{1}{\sqrt{2^n}}\sum_{z \in \{0,1\}^n} |z\rangle$, which is the uniform superposition — easy to prepare with Hadamard gates.

### 2.3 Computation Time

The total time $T$ required scales as:

$$T = O\left(\frac{1}{\Delta_{\min}^2}\right)$$

The key challenge: for hard problems, the minimum gap $\Delta_{\min}$ can close **exponentially** in the problem size $n$, making AQC take exponential time.

## 3. Problem Encoding: The Ising Model

### 3.1 QUBO and Ising Formulation

Many combinatorial optimization problems can be encoded as **Quadratic Unconstrained Binary Optimization (QUBO)**:

$$\min_{x \in \{0,1\}^n} \sum_{i} a_i x_i + \sum_{i<j} b_{ij} x_i x_j$$

Substituting $x_i = (1 - z_i)/2$ where $z_i \in \{\pm 1\}$ converts to the **Ising model**:

$$H_P = \sum_i h_i Z_i + \sum_{i<j} J_{ij} Z_i Z_j + \text{const}$$

### 3.2 Example: Max-Cut

Given a graph $G = (V, E)$, partition vertices into two sets to maximize edges between sets:

$$H_{\text{MaxCut}} = \sum_{(i,j) \in E} \frac{1}{2}(I - Z_i Z_j)$$

The ground state of $-H_{\text{MaxCut}}$ (we minimize the negative) gives the maximum cut.

### 3.3 Example: Number Partitioning

Given numbers $\{a_1, \ldots, a_n\}$, partition into two sets with equal sums:

$$H_{\text{partition}} = \left(\sum_i a_i Z_i\right)^2 = \sum_{i,j} a_i a_j Z_i Z_j$$

Ground state energy = 0 means a perfect partition exists.

In [ ]:
# Encode and solve a small Max-Cut problem using adiabatic evolution

def build_maxcut_hamiltonian(edges, n_nodes):
    """
    Build the Max-Cut Hamiltonian for a graph.
    H = sum_{(i,j) in E} (1/2)(I - Z_i Z_j)
    We minimize -H to find the maximum cut.
    """
    dim = 2**n_nodes
    H = np.zeros((dim, dim), dtype=complex)
    
    for (i, j) in edges:
        # Build Z_i Z_j operator
        ZiZj = np.eye(1)
        for k in range(n_nodes):
            if k == i or k == j:
                ZiZj = np.kron(ZiZj, sigma_z)
            else:
                ZiZj = np.kron(ZiZj, I2)
        
        H += 0.5 * (np.eye(dim) - ZiZj)
    
    return -H  # Minimize to find max cut

def build_transverse_field(n_nodes):
    """Build the transverse field Hamiltonian H_0 = -sum_i X_i."""
    dim = 2**n_nodes
    H = np.zeros((dim, dim), dtype=complex)
    
    for i in range(n_nodes):
        Xi = np.eye(1)
        for k in range(n_nodes):
            if k == i:
                Xi = np.kron(Xi, sigma_x)
            else:
                Xi = np.kron(Xi, I2)
        H -= Xi
    
    return H

# Define a small graph: triangle with extra edge
n_nodes = 4
edges = [(0, 1), (1, 2), (2, 3), (0, 3), (0, 2)]

H_P = build_maxcut_hamiltonian(edges, n_nodes)
H_0 = build_transverse_field(n_nodes)

# Find exact ground state of H_P
eigvals_P, eigvecs_P = eigh(H_P)
ground_energy = eigvals_P[0]
ground_state_idx = np.argmax(np.abs(eigvecs_P[:, 0]))

print("Max-Cut Problem")
print("=" * 40)
print(f"Graph: {n_nodes} nodes, edges = {edges}")
print(f"\nGround state energy: {ground_energy:.4f}")
print(f"Maximum cut value: {-ground_energy:.4f}")
print(f"\nAll eigenvalues: {np.round(np.real(eigvals_P), 3)}")
print(f"\nGround state(s):")

for i in range(len(eigvals_P)):
    if np.abs(eigvals_P[i] - ground_energy) < 1e-10:
        state = eigvecs_P[:, i]
        for j in range(2**n_nodes):
            if np.abs(state[j]) > 0.1:
                bitstring = format(j, f'0{n_nodes}b')
                print(f"  |{bitstring}⟩: amplitude = {state[j]:.4f}")

In [ ]:
# Run adiabatic evolution for Max-Cut

# Compute energy spectrum
s_vals, energies, gaps = compute_spectrum(H_0, H_P, n_points=300)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Energy spectrum (show first 6 levels)
ax = axes[0]
for i in range(min(6, energies.shape[1])):
    ax.plot(s_vals, energies[:, i], linewidth=1.5, label=f'E_{i}')
ax.set_xlabel('s = t/T', fontsize=12)
ax.set_ylabel('Energy', fontsize=12)
ax.set_title('Energy Spectrum', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Energy gap
ax = axes[1]
ax.plot(s_vals, gaps, 'r-', linewidth=2)
ax.axhline(y=min(gaps), color='k', linestyle='--', alpha=0.5)
ax.set_xlabel('s = t/T', fontsize=12)
ax.set_ylabel('Gap Δ(s)', fontsize=12)
ax.set_title(f'Energy Gap (Δ_min = {min(gaps):.4f})', fontsize=13)
ax.grid(True, alpha=0.3)

# Adiabatic evolution success probability
ax = axes[2]
for T in [1, 5, 10, 20, 50, 100]:
    s_sim, gs_prob = simulate_adiabatic_evolution(H_0, H_P, T, n_steps=500)
    ax.plot(s_sim, gs_prob, linewidth=1.5, label=f'T = {T}')

ax.set_xlabel('s = t/T', fontsize=12)
ax.set_ylabel('P(ground state)', fontsize=12)
ax.set_title('Ground State Fidelity', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze success probability vs evolution time T
T_values = np.logspace(-0.5, 2.5, 50)
final_probs = []

for T in T_values:
    _, gs_prob = simulate_adiabatic_evolution(H_0, H_P, T, n_steps=300)
    final_probs.append(gs_prob[-1])

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogx(T_values, final_probs, 'b-', linewidth=2)
ax.axhline(y=0.99, color='g', linestyle='--', alpha=0.5, label='99% threshold')
ax.axhline(y=0.9, color='orange', linestyle='--', alpha=0.5, label='90% threshold')

# Mark the adiabatic condition T ~ 1/Δ²
delta_min = min(gaps)
T_adiabatic = 1 / delta_min**2
ax.axvline(x=T_adiabatic, color='r', linestyle=':', alpha=0.5, 
           label=f'T ~ 1/Δ² = {T_adiabatic:.1f}')

ax.set_xlabel('Total Evolution Time T', fontsize=12)
ax.set_ylabel('Final Ground State Probability', fontsize=12)
ax.set_title('Adiabatic Success Probability vs Evolution Time', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

# Find minimum T for 99% success
for T, p in zip(T_values, final_probs):
    if p > 0.99:
        print(f"T > {T:.1f} needed for 99% success")
        print(f"Adiabatic condition estimate: T ~ 1/Δ_min² = {T_adiabatic:.1f}")
        break

## 4. Equivalence to the Gate Model

### 4.1 AQC is Universal

Aharonov et al. (2007) proved that **AQC is polynomially equivalent to the gate model**:

- Any quantum circuit of depth $D$ on $n$ qubits can be simulated by AQC in time $T = \text{poly}(D, n)$
- Any AQC with time $T$ can be simulated by a quantum circuit of size $\text{poly}(T, n)$

### 4.2 Key Idea of the Proof

**Gate model $\to$ AQC:** The Feynman clock construction encodes a quantum circuit's computation as a ground state:

$$H_{\text{clock}} = \sum_{t=0}^{T-1} \left( |t\rangle\langle t| \otimes I - |t+1\rangle\langle t| \otimes U_t - |t\rangle\langle t+1| \otimes U_t^\dagger + |t+1\rangle\langle t+1| \otimes I \right)$$

where $U_t$ is the $t$-th gate. The ground state is the **history state**:

$$|\psi_{\text{history}}\rangle = \frac{1}{\sqrt{T+1}} \sum_{t=0}^{T} |t\rangle \otimes U_t \cdots U_1 |\psi_0\rangle$$

Measuring the clock register in the final time step yields the circuit output.

### 4.3 Implications

Since AQC and the gate model are equivalent:
- AQC can solve BQP-complete problems
- Any quantum speedup in the gate model has an AQC counterpart
- The minimum gap determines the computational complexity

## 5. Quantum Annealing

### 5.1 AQC vs Quantum Annealing

**Quantum annealing** (QA) is a heuristic inspired by AQC but with key differences:

| Feature | AQC | Quantum Annealing |
|---------|-----|-------------------|
| Gap guarantee | Required | Not guaranteed |
| Temperature | $T = 0$ (ideal) | Finite temperature |
| Open system | Closed | Open (thermal bath) |
| Universality | Yes | Not proven |
| Hardware | Any | Typically Ising-type |

### 5.2 D-Wave Architecture

D-Wave quantum annealers implement the Hamiltonian:

$$H(s) = A(s) \left(-\sum_i X_i\right) + B(s) \left(\sum_i h_i Z_i + \sum_{i<j} J_{ij} Z_i Z_j\right)$$

where $A(s)$ decreases from a large value to 0, and $B(s)$ increases from 0 to a large value.

### 5.3 Annealing Schedule

The **annealing schedule** $A(s), B(s)$ can be more general than linear interpolation. Common strategies:

- **Pause and quench:** Pause near the minimum gap, then quickly anneal to the end
- **Reverse annealing:** Start from a known solution, anneal backward then forward
- **Inhomogeneous schedules:** Different rates for different qubits

In [ ]:
# Simulate quantum annealing with different schedules

def anneal_schedule_linear(s):
    """Linear schedule: A(s) = 1-s, B(s) = s"""
    return 1 - s, s

def anneal_schedule_pause(s, s_pause=0.4, pause_fraction=0.3):
    """Schedule with pause near minimum gap."""
    s_start_pause = s_pause - pause_fraction / 2
    s_end_pause = s_pause + pause_fraction / 2
    
    if s < s_start_pause:
        s_eff = s * s_start_pause / s_start_pause
    elif s < s_end_pause:
        s_eff = s_start_pause + (s - s_start_pause) * 0.01  # Nearly pause
    else:
        s_eff = s_start_pause + 0.01 * (s_end_pause - s_start_pause) + \
                (s - s_end_pause) * (1 - s_start_pause) / (1 - s_end_pause)
    
    s_eff = np.clip(s_eff, 0, 1)
    return 1 - s_eff, s_eff

def simulate_annealing(H_0, H_P, T, schedule_fn, n_steps=500):
    """Simulate quantum annealing with a custom schedule."""
    dim = H_0.shape[0]
    dt = T / n_steps
    
    eigvals, eigvecs = eigh(H_0)
    psi = eigvecs[:, 0].astype(complex)
    
    probs = []
    
    for step in range(n_steps + 1):
        s = step / n_steps
        A, B = schedule_fn(s)
        H = A * H_0 + B * H_P
        
        eigvals_cur, eigvecs_cur = eigh(H)
        prob = np.abs(np.vdot(eigvecs_cur[:, 0], psi))**2
        probs.append(prob)
        
        if step < n_steps:
            U = expm(-1j * H * dt)
            psi = U @ psi
    
    return np.linspace(0, 1, n_steps + 1), np.array(probs)

# Compare schedules
fig, ax = plt.subplots(figsize=(12, 6))

T = 15
s_lin, p_lin = simulate_annealing(H_0, H_P, T, anneal_schedule_linear)
s_pause, p_pause = simulate_annealing(H_0, H_P, T, anneal_schedule_pause)

ax.plot(s_lin, p_lin, 'b-', linewidth=2, label=f'Linear schedule (P_final={p_lin[-1]:.4f})')
ax.plot(s_pause, p_pause, 'r-', linewidth=2, label=f'Pause schedule (P_final={p_pause[-1]:.4f})')

ax.set_xlabel('s = t/T', fontsize=12)
ax.set_ylabel('Ground State Probability', fontsize=12)
ax.set_title(f'Annealing Schedule Comparison (T={T})', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 6. Energy Gap Analysis

### 6.1 Gap Scaling and Computational Hardness

The minimum energy gap determines the computational complexity:

| Gap Scaling | AQC Time | Complexity |
|------------|----------|------------|
| $\Delta_{\min} = O(1)$ | $O(1)$ | Easy |
| $\Delta_{\min} = O(1/\text{poly}(n))$ | $O(\text{poly}(n))$ | Efficient |
| $\Delta_{\min} = O(e^{-cn})$ | $O(e^{2cn})$ | Exponential |

### 6.2 First-Order vs Second-Order Phase Transitions

- **First-order transition:** Gap closes exponentially $\Rightarrow$ AQC is exponentially slow
- **Second-order transition:** Gap closes polynomially $\Rightarrow$ AQC may be efficient

Many NP-hard problems exhibit first-order transitions, explaining why AQC doesn't easily solve NP-hard problems.

In [ ]:
# Energy gap scaling analysis for different problem sizes

def random_maxcut_gap(n_nodes, n_edges_frac=0.5, seed=None):
    """Compute minimum gap for random Max-Cut instance."""
    if seed is not None:
        np.random.seed(seed)
    
    # Random graph
    edges = []
    for i in range(n_nodes):
        for j in range(i+1, n_nodes):
            if np.random.random() < n_edges_frac:
                edges.append((i, j))
    
    if len(edges) == 0:
        edges = [(0, 1)]  # At least one edge
    
    H_P = build_maxcut_hamiltonian(edges, n_nodes)
    H_0 = build_transverse_field(n_nodes)
    
    # Compute minimum gap
    min_gap = np.inf
    for s in np.linspace(0, 1, 200):
        H = (1 - s) * H_0 + s * H_P
        eigvals = np.sort(np.real(np.linalg.eigvalsh(H)))
        gap = eigvals[1] - eigvals[0]
        min_gap = min(min_gap, gap)
    
    return min_gap

# Compute gaps for different sizes
sizes = [2, 3, 4, 5, 6, 7]
n_instances = 10

print("Computing minimum gaps for random Max-Cut instances...")
avg_gaps = []
std_gaps = []

for n in sizes:
    gaps_n = []
    for seed in range(n_instances):
        gap = random_maxcut_gap(n, n_edges_frac=0.5, seed=seed + n*100)
        gaps_n.append(gap)
    avg_gaps.append(np.mean(gaps_n))
    std_gaps.append(np.std(gaps_n))
    print(f"  n={n}: Δ_min = {np.mean(gaps_n):.4f} ± {np.std(gaps_n):.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.errorbar(sizes, avg_gaps, yerr=std_gaps, fmt='bo-', linewidth=2, capsize=5)
ax1.set_xlabel('Number of Nodes n', fontsize=12)
ax1.set_ylabel('Average Minimum Gap', fontsize=12)
ax1.set_title('Minimum Gap vs Problem Size', fontsize=13)
ax1.grid(True, alpha=0.3)

# Log scale to check exponential vs polynomial
ax2.semilogy(sizes, avg_gaps, 'bo-', linewidth=2)
ax2.set_xlabel('Number of Nodes n', fontsize=12)
ax2.set_ylabel('Average Minimum Gap (log scale)', fontsize=12)
ax2.set_title('Gap Scaling (Log Scale)', fontsize=13)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nNote: For these small sizes, the gap behavior is dominated by finite-size effects.")
print("Asymptotic exponential closing typically appears for n >> 10.")

## 7. Solving a Number Partitioning Problem

Let us solve a concrete optimization problem: partition the set $\{3, 1, 4, 1, 5\}$ into two subsets with equal (or closest) sums.

In [ ]:
# Number partitioning problem
numbers = [3, 1, 4, 1, 5]
n = len(numbers)

def build_partition_hamiltonian(numbers):
    """Build H = (sum_i a_i Z_i)^2 for number partitioning."""
    n = len(numbers)
    dim = 2**n
    H = np.zeros((dim, dim), dtype=complex)
    
    # Build sum_i a_i Z_i
    S = np.zeros((dim, dim), dtype=complex)
    for i in range(n):
        Zi = np.eye(1)
        for k in range(n):
            if k == i:
                Zi = np.kron(Zi, sigma_z)
            else:
                Zi = np.kron(Zi, I2)
        S += numbers[i] * Zi
    
    H = S @ S  # H = S^2
    return H

H_P_part = build_partition_hamiltonian(numbers)
H_0_part = build_transverse_field(n)

# Exact solution
eigvals, eigvecs = eigh(H_P_part)
print(f"Numbers to partition: {numbers}")
print(f"Total sum: {sum(numbers)}")
print(f"\nGround state energy: {eigvals[0]:.4f}")
print(f"(Energy 0 = perfect partition, >0 = best achievable)")

# Find all ground states
ground_e = eigvals[0]
print(f"\nOptimal partitions:")
for j in range(2**n):
    if np.abs(eigvecs[j, 0]) > 0.1:
        bits = format(j, f'0{n}b')
        set_a = [numbers[i] for i in range(n) if bits[i] == '0']
        set_b = [numbers[i] for i in range(n) if bits[i] == '1']
        print(f"  {bits}: Set A = {set_a} (sum={sum(set_a)}), Set B = {set_b} (sum={sum(set_b)})")

In [ ]:
# Solve with adiabatic evolution
s_vals_p, energies_p, gaps_p = compute_spectrum(H_0_part, H_P_part, n_points=300)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Spectrum
ax = axes[0]
for i in range(min(8, energies_p.shape[1])):
    ax.plot(s_vals_p, energies_p[:, i], linewidth=1, alpha=0.8)
ax.set_xlabel('s', fontsize=12)
ax.set_ylabel('Energy', fontsize=12)
ax.set_title('Energy Spectrum (Number Partitioning)', fontsize=12)
ax.grid(True, alpha=0.3)

# Gap
ax = axes[1]
ax.plot(s_vals_p, gaps_p, 'r-', linewidth=2)
ax.set_xlabel('s', fontsize=12)
ax.set_ylabel('Gap', fontsize=12)
ax.set_title(f'Energy Gap (min = {min(gaps_p):.4f})', fontsize=12)
ax.grid(True, alpha=0.3)

# Evolution
ax = axes[2]
for T in [5, 20, 50, 100, 200]:
    s_sim, gs_prob = simulate_adiabatic_evolution(H_0_part, H_P_part, T, n_steps=400)
    ax.plot(s_sim, gs_prob, linewidth=1.5, label=f'T={T} (P_f={gs_prob[-1]:.3f})')

ax.set_xlabel('s', fontsize=12)
ax.set_ylabel('P(ground state)', fontsize=12)
ax.set_title('Adiabatic Evolution', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

## 8. Landau-Zener Transitions

### 8.1 The Landau-Zener Formula

When the energy gap closes, the probability of a **diabatic transition** (leaving the ground state) is given by the Landau-Zener formula:

$$P_{\text{transition}} = e^{-\pi \Delta_{\min}^2 T / (2|\alpha|)}$$

where $\alpha = \frac{d}{dt}(E_1 - E_0)$ is the rate of change of the gap near the minimum. This gives us the **ground state probability**:

$$P_{\text{ground}} = 1 - e^{-\pi \Delta_{\min}^2 T / (2|\alpha|)}$$

### 8.2 Implications

- The success probability increases **exponentially** with $\Delta_{\min}^2 T$
- If $\Delta_{\min} \sim e^{-n}$, then $T \sim e^{2n}$ is needed for high probability
- This is the fundamental reason first-order phase transitions make AQC inefficient

In [ ]:
# Landau-Zener analysis
# Verify the LZ formula for our 2-level system

H_i_2 = -sigma_x
H_f_2 = -sigma_z

# Compute minimum gap and slope
s_vals_lz, energies_lz, gaps_lz = compute_spectrum(H_i_2, H_f_2, n_points=1000)
delta_min_2 = min(gaps_lz)
min_idx = np.argmin(gaps_lz)

# Estimate alpha (slope of gap near minimum)
ds = s_vals_lz[1] - s_vals_lz[0]
dgap_ds = np.gradient(gaps_lz, ds)
alpha = np.abs(dgap_ds[min_idx]) if np.abs(dgap_ds[min_idx]) > 1e-10 else 1.0

# Simulate for many T values and compare with LZ prediction
T_vals = np.linspace(0.5, 30, 40)
P_sim = []
P_lz = []

for T in T_vals:
    _, gs = simulate_adiabatic_evolution(H_i_2, H_f_2, T, n_steps=500)
    P_sim.append(gs[-1])
    P_lz.append(1 - np.exp(-np.pi * delta_min_2**2 * T / 2))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(T_vals, P_sim, 'bo', markersize=6, label='Simulation')
ax.plot(T_vals, P_lz, 'r-', linewidth=2, label='Landau-Zener prediction')
ax.set_xlabel('Total Evolution Time T', fontsize=12)
ax.set_ylabel('Ground State Probability', fontsize=12)
ax.set_title('Landau-Zener Transition: Simulation vs Theory', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print(f"Minimum gap: Δ_min = {delta_min_2:.4f}")
print(f"The Landau-Zener formula provides a good approximation of the transition probability.")

## 9. Summary and Key Takeaways

### What We Learned

| Concept | Key Result |
|---------|------------|
| Adiabatic theorem | Slow evolution keeps system in ground state |
| Time bound | $T \gg \|dH/ds\| / \Delta_{\min}^2$ |
| AQC framework | Encode solution as ground state of $H_P$ |
| Ising encoding | QUBO $\to$ $H = \sum h_i Z_i + \sum J_{ij} Z_i Z_j$ |
| Universality | AQC $\equiv$ gate model (polynomial overhead) |
| Quantum annealing | Heuristic version; D-Wave hardware |
| Gap scaling | $\Delta_{\min} \sim e^{-n}$ for hard problems |
| Landau-Zener | $P_{\text{fail}} = e^{-\pi\Delta^2 T / (2\alpha)}$ |

### Looking Ahead

In the next notebook, we will explore **Quantum Machine Learning**, where quantum circuits are used to process classical data and build quantum classifiers.

---

*Notebook 13 of the Quantum Computing from Ground Up series.*